# Lesson 6: Essay Writer

## 本节知识点概括

这一节用 `Essay Writer` 这个例子，把前面学到的状态、分步执行、反思和循环整合成一个更完整的多阶段 Agent 工作流。

你会在这个 notebook 里掌握这些核心点：

- 如何把一个复杂任务拆成多个阶段：规划、检索、写作、批评、修订
- 为什么复杂 Agent 不能只靠一次提示词：因为高质量输出通常需要多轮迭代
- `plan -> draft -> critique -> revise` 这种循环为什么重要：它把“先做、再评、再改”显式变成图结构
- 如何用状态对象保存草稿、批评意见、研究材料、修订次数等中间产物
- 为什么 LangGraph 特别适合这种长流程任务：因为它天然支持节点、循环和条件终止

这一节本质上是在教你：
- 如何把一个复杂认知任务工程化
- 如何让 Agent 拥有更像“工作流”而不是“单轮问答”的行为模式


In [1]:
from dotenv import load_dotenv
import os

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 建立内存数据库连接，用于保存agent状态
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

In [3]:
# 定义代理状态类型，使用 TypedDict 确保类型安全
class AgentState(TypedDict):
    task: str  # 用户请求的论文主题
    plan: str  # 论文大纲/计划
    draft: str  # 论文草稿
    critique: str  # 对草稿的批评意见
    content: List[str]  # 研究收集的内容列表
    revision_number: int  # 当前修订次数
    max_revisions: int  # 最大允许修订次数

In [4]:
from langchain_community.chat_models.tongyi import ChatTongyi
model = ChatTongyi(
    model="qwen-max",  # 或其他通义千问模型
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),  # 通义千问 API key
    temperature=0.1, 
    streaming=True
)

In [5]:
# 规划提示词：指导模型生成论文大纲
# 提示词翻译：
# 您是一位经验丰富的写手，任务是撰写一篇论文的高级提纲。
# 请根据用户提供的主题撰写一份提纲。提纲应包含文章的概要以及各部分相关的注释或说明。
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
or instructions for the sections."""

In [6]:
# 写作提示词：指导模型根据大纲和研究内容撰写论文
# 提示词翻译：
# 你是一名论文助教，任务是撰写优秀的五段式论文。
# 根据用户的要求和初始提纲，生成尽可能最好的论文。
# 如果用户提出批评意见，请回复修改后的版本。
# 请根据需要使用以下所有信息：
WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
Generate the best essay possible for the user's request and the initial outline. \
If the user provides critique, respond with a revised version of your previous attempts. \
Utilize all the information below as needed: 

------

{content}"""

In [7]:
# 反思提示词：指导模型作为教师角色对论文进行评价
# 提示词翻译：
# 你是一名老师，正在批改一篇作文。
# 对用户提交的内容进行评价和提出建议。
# 提供详细的建议，包括长度、深度、款式等要求。
REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
Generate critique and recommendations for the user's submission. \
Provide detailed recommendations, including requests for length, depth, style, etc."""

In [8]:
# 研究规划提示词：指导模型生成搜索查询以收集论文所需信息
# 提示词翻译：
# 你是一名研究员，负责提供撰写以下论文所需的信息。
# 请生成一份搜索查询列表，以收集所有相关信息。最多只能生成 3 个查询。
RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can \
be used when writing the following essay. Generate a list of search queries that will gather \
any relevant information. Only generate 3 queries max."""


In [9]:
# 研究批评提示词：指导模型生成搜索查询以改进论文
# 提示词翻译：
# 您是一名研究人员，负责提供信息，以便在进行任何要求的修改时使用（如下所述）。
# 生成一系列搜索查询，用于收集所有相关信息。最多只能生成 3 个查询。
RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can \
be used when making any requested revisions (as outlined below). \
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


In [10]:
from pydantic import BaseModel

# 定义查询模型，确保输出是包含查询列表的对象
class Queries(BaseModel):
    queries: List[str]

In [11]:
from tavily import TavilyClient
import os
# 使用环境变量中的 API 密钥初始化 Tavily 客户端
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [12]:
def plan_node(state: AgentState):
    """规划节点：生成论文大纲
    
    Args:
        state: 当前代理状态，包含用户任务
        
    Returns:
        包含生成的论文大纲的字典
    """
    # 构建消息列表：系统提示 + 用户任务
    messages = [
        SystemMessage(content=PLAN_PROMPT),  # 系统提示定义角色和任务
        HumanMessage(content=state['task'])  # 用户提供的论文主题
    ]
    # 调用模型生成响应
    response = model.invoke(messages)
    # 返回生成的论文大纲
    return {"plan": response.content}

In [13]:
def research_plan_node(state: AgentState):
    """研究规划节点：为论文大纲收集相关信息
    
    Args:
        state: 当前代理状态，包含任务和可能已有的内容
        
    Returns:
        包含新收集内容的字典
    """
    # 使用结构化输出，让模型生成查询列表
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    
    # 获取已有内容或初始化空列表
    content = state.get('content', [])
    
    # 对每个查询执行搜索并收集结果
    for q in queries.queries:
        # 执行搜索，最多返回 2 个结果
        response = tavily.search(query=q, max_results=2)
        # 将每个搜索结果的内容添加到内容列表
        for r in response['results']:
            content.append(r['content'])
    
    # 返回更新后的内容列表
    return {"content": content}

In [14]:
def generation_node(state: AgentState):
    """生成节点：基于大纲和研究内容撰写论文草稿
    
    Args:
        state: 当前代理状态
        
    Returns:
        包含草稿和更新后的修订次数的字典
    """
    # 将研究内容连接成一个字符串
    content = "\n\n".join(state['content'] or [])

    # 创建用户消息，包含任务和计划
    user_message = HumanMessage(content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    # 构建完整的消息列表
    messages = [
        SystemMessage(
            # 将研究内容插入到写作提示中
            content=WRITER_PROMPT.format(content=content)
        ),
        user_message
        ]
    # 调用模型生成论文草稿
    response = model.invoke(messages)
    # 返回草稿并增加修订次数
    return {
        "draft": response.content, 
        "revision_number": state.get("revision_number", 1) + 1
    }


In [15]:
def reflection_node(state: AgentState):
    """反思节点：对论文草稿进行评价
    
    Args:
        state: 当前代理状态，包含论文草稿
        
    Returns:
        包含批评意见的字典
    """
    # 构建消息列表：系统提示 + 论文草稿
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),  # 系统提示定义评价角色
        HumanMessage(content=state['draft'])  # 需要评价的论文草稿
    ]
    # 调用模型生成批评意见
    response = model.invoke(messages)
    # 返回批评意见
    return {"critique": response.content}

In [16]:
def research_critique_node(state: AgentState):
    """研究批评节点：为改进论文收集相关信息
    
    Args:
        state: 当前代理状态，包含批评意见
        
    Returns:
        包含新收集内容的字典
    """
    # 使用结构化输出，让模型基于批评意见生成查询列表
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])

    # 获取已有内容或初始化空列表
    content = state.get('content', [])
    # 对每个查询执行搜索并收集结果
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    
    # 返回更新后的内容列表
    return {"content": content}

In [17]:
def should_continue(state):
    """条件函数：判断是否需要继续修订
    
    Args:
        state: 当前代理状态
        
    Returns:
        如果达到最大修订次数则返回 END，否则返回 "reflect" 继续修订
    """
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

In [18]:
# 创建状态图，使用 AgentState 作为状态类型
builder = StateGraph(AgentState)

In [19]:
# 添加各个节点到状态图
builder.add_node("planner", plan_node)  # 规划节点
builder.add_node("generate", generation_node)  # 生成节点
builder.add_node("reflect", reflection_node)  # 反思节点
builder.add_node("research_plan", research_plan_node)  # 研究规划节点
builder.add_node("research_critique", research_critique_node)  # 研究批评节点

In [20]:
# 设置入口点为规划节点
builder.set_entry_point("planner")

In [21]:
# 添加条件边：从生成节点根据 should_continue 函数决定下一步
builder.add_conditional_edges(
    "generate", 
    should_continue, 
    {END: END, "reflect": "reflect"}
)


In [22]:
# 添加常规边：定义节点之间的固定流程
builder.add_edge("planner", "research_plan")  # 规划后进行研究
builder.add_edge("research_plan", "generate")  # 研究后生成草稿

builder.add_edge("reflect", "research_critique")  # 反思后进行针对性研究
builder.add_edge("research_critique", "generate")  # 研究后重新生成


In [23]:
# 编译状态图，添加检查点保存器以便恢复状态
graph = builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image
# 用图片方式展示流程图
png_data=graph.graph.get_graph().draw_mermaid_png()
Image(png_data)
# graph.get_graph().print_ascii()

                     +-----------+                  
                     | __start__ |                  
                     +-----------+                  
                            *                       
                            *                       
                            *                       
                      +---------+                   
                      | planner |                   
                      +---------+                   
                            *                       
                            *                       
                            *                       
                   +---------------+                
                   | research_plan |                
                   +---------------+                
                            *                       
                            *                       
                            *                       
                      +----------+            

In [25]:
# 定义线程配置，每个对话有唯一的 thread_id
thread = {"configurable": {"thread_id": "1"}}
# 流式执行图，传入初始状态
for s in graph.stream({
    'task': "what is the difference between langchain and langsmith",
    "max_revisions": 2,
    "revision_number": 1,
}, thread):
    print(s)

{'planner': {'plan': '### Essay Outline: Differences Between LangChain and LangSmith\n\n#### I. Introduction\n   - **A.** Brief overview of the importance of language models in AI.\n   - **B.** Introduction to LangChain and LangSmith.\n   - **C.** Thesis statement: This essay will explore the key differences between LangChain and LangSmith, highlighting their unique features, use cases, and implications for developers and businesses.\n\n#### II. Background on LangChain\n   - **A.** Definition and purpose of LangChain.\n   - **B.** Key features and capabilities.\n     - 1. Integration with various language models.\n     - 2. Modular architecture.\n     - 3. Customizable workflows.\n   - **C.** Target audience and typical use cases.\n   - **D.** Development and maintenance.\n   - **E.** Community and support.\n\n#### III. Background on LangSmith\n   - **A.** Definition and purpose of LangSmith.\n   - **B.** Key features and capabilities.\n     - 1. Specialized focus (e.g., fine-tuning, d

## Essay Writer Interface 论文写作界面

In [8]:
import warnings
warnings.filterwarnings("ignore")

import importlib
import helper

importlib.reload(helper)

from helper import ewriter, writer_gui

In [11]:

# 创建多代理论文写作者
MultiAgent = ewriter()
# 创建GUI应用
app = writer_gui(MultiAgent.graph)
# 启动GUI应用
app.launch()

* Running on local URL:  http://127.0.0.1:7865

To create a public link, set `share=True` in `launch()`.
